### ЗАДАЧА: Панель обработки security-инцидентов по паттерну `MVC`

Внутренняя SOC-команда ведет инциденты в консольной панели.
Для каждого инцидента нужно хранить сервис, severity, статус,
аналитика и краткое описание.

Нужно реализовать систему по паттерну `MVC`, где:
- `Model` хранит инциденты и правила переходов статусов,
- `View` отвечает только за отображение,
- `Controller` принимает действия и связывает `Model` и `View`.

НЕОБХОДИМО:

1. Реализовать сущность `SecurityIncident`.
2. Реализовать `IncidentModel`, который умеет:
   - добавлять инцидент,
   - назначать аналитика,
   - менять статус инцидента,
   - возвращать список инцидентов для отображения.
3. Бизнес-правила модели:
   - инцидент нельзя перевести в `investigating` без аналитика,
   - статус можно менять только по цепочке:
     `new -> investigating -> contained -> resolved`,
   - перевод в `resolved` возможен только из `contained`.
4. Реализовать `IncidentView` для списка, успешных сообщений и ошибок.
5. Реализовать `IncidentController`, который вызывает методы модели и передает результат во view.
6. Загрузить начальные инциденты из `initial_incidents` в модель.
7. Обработать все действия из `actions` через `controller`.
8. В конце вывести финальное состояние панели инцидентов.


In [ ]:
from dataclasses import dataclass


initial_incidents = [
    ("INC-500", "payment-gateway", "high", "Suspicious payout pattern"),
    ("INC-501", "auth-service", "medium", "Too many failed logins"),
]

actions = [
    ("show",),
    ("assign", "INC-500", "Nikita"),
    ("status", "INC-500", "investigating"),
    ("status", "INC-500", "contained"),
    ("status", "INC-500", "resolved"),
    ("create", "INC-502", "order-service", "critical", "Mass order creation from same IP"),
    ("status", "INC-502", "investigating"),
    ("assign", "INC-502", "Olga"),
    ("status", "INC-502", "investigating"),
    ("status", "INC-502", "resolved"),
    ("status", "INC-502", "contained"),
    ("status", "INC-502", "resolved"),
    ("show",),
]


@dataclass
class SecurityIncident:
    incident_id: str
    service: str
    severity: str
    title: str
    status: str = "new"
    analyst: str | None = None


class IncidentModel:
    allowed_transitions = {
        "new": {"investigating"},
        "investigating": {"contained"},
        "contained": {"resolved"},
        "resolved": set(),
    }

    def __init__(self):
        # TODO 1: создайте self.incidents как пустой словарь
        self.incidents = {}

    def add_incident(self, incident_id: str, service: str, severity: str, title: str) -> None:
        # TODO 2: если incident_id уже есть, вызовите ValueError('Incident already exists')
        if incident_id in self.incidents:
            raise ValueError('Incident already exists')
        # TODO 3: создайте SecurityIncident и сохраните в self.incidents
        self.incidents[incident_id] = SecurityIncident(incident_id, service, severity, title)

    def assign_analyst(self, incident_id: str, analyst: str) -> None:
        # TODO 4: если инцидент не найден, вызовите ValueError('Incident not found')
        if incident_id not in self.incidents:
            raise ValueError('Incident not found')
        # TODO 5: назначьте analyst
        self.incidents[incident_id].analyst = analyst

    def change_status(self, incident_id: str, new_status: str) -> None:
        # TODO 6: если инцидент не найден, вызовите ValueError('Incident not found')
        if incident_id not in self.incidents:
            raise ValueError('Incident not found')
        
        incident = self.incidents[incident_id]
        # TODO 7: если new_status == 'investigating' и analyst не назначен,
        # вызовите ValueError('Analyst is required')
        if new_status == 'investigating' and incident.analyst is None:
            raise ValueError('Analyst is required')

        # TODO 8: проверьте допустимость перехода через allowed_transitions
        current_status = incident.status
        allowed_next_statuses = self.allowed_transitions.get(current_status, set())
        # TODO 9: если переход недопустим, вызовите ValueError('Invalid status transition')
        if new_status not in allowed_next_statuses:
            raise ValueError('Invalid status transition')
        # TODO 10: обновите статус
        incident.status = new_status

    def list_incidents(self) -> list[str]:
        # TODO 11: верните список строк формата:
        #   'incident_id | severity | status | analyst | service | title'
        result = []
        for incident in self.incidents.values():
            analyst_display = incident.analyst if incident.analyst else "None"
            row = f"{incident.incident_id} | {incident.severity} | {incident.status} | {analyst_display} | {incident.service} | {incident.title}"
            result.append(row)
        return result


class IncidentView:
    @staticmethod
    def render_incidents(rows: list[str]) -> None:
        # TODO 12: если rows пустой, выведите 'Инцидентов нет'
        if not rows:
            print('Инцидентов нет')
        # TODO 13: иначе выведите заголовок 'Панель инцидентов:' и затем все строки
        else:
            print('Панель инцидентов:')
            for row in rows:
                print(row)


    @staticmethod
    def render_success(message: str) -> None:
        # TODO 14: выведите сообщение успеха
        print(f"Успех: {message}")

    @staticmethod
    def render_error(message: str) -> None:
        # TODO 15: выведите сообщение ошибки
        print(f"Ошибка: {message}")


class IncidentController:
    def __init__(self, model: IncidentModel, view: IncidentView):
        # TODO 16: сохраните model и view
        self.model = model
        self.view = view

    def create_incident(self, incident_id: str, service: str, severity: str, title: str) -> None:
        # TODO 17: через try/except вызовите model.add_incident(...)
        # TODO 18: покажите success или error
        try:
            self.model.add_incident(incident_id, service, severity, title)
            self.view.render_success(f"Инцидент {incident_id} создан")
        except ValueError as e:
            self.view.render_error(str(e))

    def assign_analyst(self, incident_id: str, analyst: str) -> None:
        # TODO 19: через try/except вызовите model.assign_analyst(...)
        # TODO 20: покажите success или error
        try:
            self.model.assign_analyst(incident_id, analyst)
            self.view.render_success(f"Аналитик {analyst} назначен на инцидент {incident_id}")
        except ValueError as e:
            self.view.render_error(str(e))

    def change_status(self, incident_id: str, new_status: str) -> None:
        # TODO 21: через try/except вызовите model.change_status(...)
        # TODO 22: покажите success или error
        try:
            self.model.change_status(incident_id, new_status)
            self.view.render_success(f"Статус инцидента {incident_id} изменён на {new_status}")
        except ValueError as e:
            self.view.render_error(str(e))

    def show_incidents(self) -> None:
        # TODO 23: получите данные из model.list_incidents() и передайте их во view.render_incidents(...)
        rows = self.model.list_incidents()
        self.view.render_incidents(rows)


model = IncidentModel()
view = IncidentView()
controller = IncidentController(model, view)

# TODO 24: загрузите начальные инциденты из initial_incidents в model
# Подсказка: пройдите по initial_incidents циклом for
# и вызовите model.add_incident(incident_id, service, severity, title)
for incident_data in initial_incidents:
    incident_id, service, severity, title = incident_data
    controller.create_incident(incident_id, service, severity, title)

# TODO 25: обработайте все действия из actions
for action in actions:
    action_type = action[0]
# TODO 26: если action[0] == 'show', вызовите controller.show_incidents()
if action_type == 'show':
    controller.show_incidents()
# TODO 27: если action[0] == 'create', распакуйте incident_id, service, severity, title
# и вызовите controller.create_incident(...)
elif action_type == 'create':
    _, incident_id, service, severity, title = action
# TODO 28: если action[0] == 'assign', распакуйте incident_id и analyst
# и вызовите controller.assign_analyst(...)
elif action_type == 'assign':
    _, incident_id, analyst = action
    controller.assign_analyst(incident_id, analyst)

# TODO 29: если action[0] == 'status', распакуйте incident_id и new_status
# и вызовите controller.change_status(...)
elif action_type == 'status':
    _, incident_id, new_status = action
    controller.change_status(incident_id, new_status)

print()
print("Финальное состояние панели инцидентов:")
# TODO 30: покажите финальное состояние панели через controller.show_incidents()
controller.show_incidents()